# Groningen Short-Stack Radar-Coding Example

This notebook runs the Groningen short-stack example with three explicit steps:

1. `selection.py groningen.parms`
2. `mainRC.py RC.parms`
3. `mainALE.py ALE.parms`

The example uses the short cropped Sentinel-1 descending track 037 stack located under:

```text
example/aoi_groningen/nl_groningen_s1_dsc_t037_short/cropped_stack/
```


## Required Inputs

The root configuration files for this example are:

```text
groningen.parms
RC.parms
ALE.parms
stacks_RC.json
```

The target database is:

```text
Database/Database_DesignatedTargets_TUDelft_v20240320.xlsx
```

The cropped stack folder must contain:

```text
cropped_stack/
├── nl_groningen_shape.shp
├── nl_groningen_shape.shx
├── nl_groningen_shape.dbf
├── nl_groningen_shape.prj
├── nlines_crp.txt
├── npixels_crp.txt
├── 20200328/
│   └── master.res
├── 20201024/
│   ├── slave.res
│   └── slc_srd.raw
├── 20201030/
│   ├── slave.res
│   └── slc_srd.raw
└── 20201105/
    ├── slave.res
    └── slc_srd.raw
```

`stackburst_coverage.*` is optional for this cropped-stack workflow. If it is available one level above `cropped_stack/`, it is used for the footprint plot. If it is missing, the code prints a warning and plots the bounding box of `nl_groningen_shape.shp` instead.


## Step 1: Select Designated Targets

`selection.py` checks which targets from the target database fall inside the AOI shapefile:

```text
cropped_stack/nl_groningen_shape.shp
```

It is configured by:

```text
groningen.parms
```

Expected outputs:

```text
results/nl_groningen_s1_dsc_t037_short/reflectors.json
results/nl_groningen_s1_dsc_t037_short/reflectors.csv
results/nl_groningen_s1_dsc_t037_short/EPSG_4258_coordinates.csv
results/nl_groningen_s1_dsc_t037_short/SelectedTargetsMap.geojson
```

`reflectors.json` is the main input for the next two steps.


In [ ]:
import subprocess

command = ["python", "selection.py", "groningen.parms"]
result = subprocess.run(command, capture_output=True, text=True)

print("Output:\n", result.stdout)
print("Errors:\n", result.stderr)
print("Return code:", result.returncode)


## Step 2: Radar Coding Export

`mainRC.py` predicts radar coordinates for the selected reflectors and exports an STM-like CSV for downstream DePSI use.

It is configured by:

```text
RC.parms
stacks_RC.json
```

Main inputs:

```text
results/nl_groningen_s1_dsc_t037_short/reflectors.json
example/aoi_groningen/nl_groningen_s1_dsc_t037_short/cropped_stack/
```

Expected main output:

```text
results/nl_groningen_s1_dsc_t037_short/RadarCoordinates/s1_dsc037_RC.csv
```

The CSV contains target ID, validation flag, radar range/azimuth, geographic coordinates, height, and one `0/1` column per SLC date. If a reflector is selected but not identified in any SLC, it remains in the CSV with `nan` radar coordinates and `0` in all date columns.


In [ ]:
import subprocess

command = ["python", "mainRC.py", "RC.parms"]
result = subprocess.run(command, capture_output=True, text=True)

print("Output:\n", result.stdout)
print("Errors:\n", result.stderr)
print("Return code:", result.returncode)


In [ ]:
from pathlib import Path

rc_csv = Path("results/nl_groningen_s1_dsc_t037_short/RadarCoordinates/s1_dsc037_RC.csv")
if rc_csv.exists():
    print("Radar-coordinate CSV:", rc_csv)
    for line in rc_csv.read_text().splitlines():
        if line.startswith("POST,"):
            print("POST row:", line)
            break
else:
    print("Radar-coordinate CSV not found:", rc_csv)


## Step 3: ALE and Reflector Quality Analysis

`mainALE.py` performs reflector localization and quality analysis using the selected targets and the short cropped stack.

It is configured by:

```text
ALE.parms
stacks_RC.json
```

For this cropped coregistered short-stack example, `ALE.parms` uses:

```text
cropFlag = 1
ovsFactor = 1
```

`cropFlag = 1` tells ALE to read the cropped-stack metadata and `slc_srd.raw` files. `ovsFactor = 1` is used because the current coregistered-stack ALE path fills measurements in the non-oversampled branch.

Expected outputs:

```text
results/nl_groningen_s1_dsc_t037_short_ALE/*_timeseries.txt
results/nl_groningen_s1_dsc_t037_short_ALE/*_stats.json
results/nl_groningen_s1_dsc_t037_short_ALE/*_ALE_TS.png
results/nl_groningen_s1_dsc_t037_short_ALE/*_RCS_TS.png
results/nl_groningen_s1_dsc_t037_short_ALE/network_ALE.png
results/nl_groningen_s1_dsc_t037_short_ALE/network_RCS.png
results/nl_groningen_s1_dsc_t037_short_ALE/network_SCR.png
```


In [ ]:
import subprocess

command = ["python", "mainALE.py", "ALE.parms"]
result = subprocess.run(command, capture_output=True, text=True)

print("Output:\n", result.stdout)
print("Errors:\n", result.stderr)
print("Return code:", result.returncode)


In [ ]:
from pathlib import Path

ale_out = Path("results/nl_groningen_s1_dsc_t037_short_ALE")
if ale_out.exists():
    print("ALE output folder:", ale_out)
    print("Station stats JSON files:", len(list(ale_out.glob("*_stats.json"))))
    print("Station time-series files:", len(list(ale_out.glob("*_timeseries.txt"))))
    print("PNG files:", len(list(ale_out.glob("*.png"))))
    for name in ["network_ALE.png", "network_RCS.png", "network_SCR.png"]:
        print(name, "exists:", (ale_out / name).exists())
else:
    print("ALE output folder not found:", ale_out)


## Summary

After the three steps:

```text
selection.py
  creates the selected target list.

mainRC.py
  creates the radar-coordinate CSV used as STM-like input.

mainALE.py
  creates localization, RCS, SCR, station-level, and network diagnostic outputs.
```

For DePSI ingestion, the most important file is:

```text
results/nl_groningen_s1_dsc_t037_short/RadarCoordinates/s1_dsc037_RC.csv
```
